In [1]:
# Instalamos python-dotenv para poder cargar la API key de forma segura desde el archivo .env
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
#KARENS_API_KEY

# Importamos las herramientas necesarias para cargar y leer las variables del archivo .env
from dotenv import load_dotenv
import os

# Cargamos las variables guardadas en el archivo .env
load_dotenv()

# Guardamos la API key en una variable para poder utilizarla más adelante
api_key = os.getenv("KARENS_API_KEY")

In [3]:
# Comprobamos que la API key se ha cargado correctamente sin mostrar su contenido
if api_key:
    print("API key cargada correctamente")
else:
    print("No se ha podido cargar la API key")

API key cargada correctamente


In [4]:
# Importamos requests para poder realizar peticiones HTTP a la API
import requests

# Definimos el endpoint de The Cat API que utilizaremos para consultar las razas
url = "https://api.thecatapi.com/v1/breeds"

In [5]:
# Creamos los headers para enviar nuestra API key de forma segura en la petición
headers = {
    "x-api-key": api_key
}

# Realizamos una petición GET al endpoint de razas
response = requests.get(url, headers=headers)

# Comprobamos el código de estado de la respuesta
print(response.status_code)

200


In [6]:
# Convertimos la respuesta de la API de formato JSON a un objeto de Python
data = response.json()

# Comprobamos qué tipo de objeto hemos recibido
print(type(data))

# Comprobamos cuántos registros hemos obtenido
print(len(data))

<class 'list'>
107


In [7]:
import pandas as pd

In [8]:
# Convertimos los datos obtenidos de la API en un DataFrame
df_cats = pd.DataFrame(data)

# Comprobamos las dimensiones del DataFrame
print(df_cats.shape)

(107, 18)


In [9]:
# Comprobamos la estructura del DataFrame y los tipos de datos de cada columna
df_cats.info()

<class 'pandas.DataFrame'>
RangeIndex: 107 entries, 0 to 106
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   id                  107 non-null    str   
 1   name                107 non-null    str   
 2   species_id          107 non-null    str   
 3   life_span           107 non-null    str   
 4   temperament         107 non-null    str   
 5   origin              100 non-null    str   
 6   country_codes       100 non-null    str   
 7   country_code        100 non-null    str   
 8   description         107 non-null    str   
 9   bred_for            0 non-null      object
 10  perfect_for         0 non-null      object
 11  breed_group         100 non-null    str   
 12  history             100 non-null    str   
 13  alt_names           55 non-null     str   
 14  reference_image_id  66 non-null     str   
 15  weight              107 non-null    object
 16  height              107 non-null    o

In [10]:
# Seleccionamos las columnas que nos interesan para el análisis
columns_analysis = [
    "name",
    "life_span",
    "temperament",
    "origin",
    "country_codes",
    "country_code",
    "description",
    "breed_group",
    "weight",
    "height"
]

# Creamos un nuevo DataFrame solo con las variables seleccionadas
df_cat_analysis = df_cats[columns_analysis].copy()

# Mostramos las primeras filas
df_cat_analysis.head()

,name,life_span,temperament,origin,country_codes,country_code,description,breed_group,weight,height
0,Abyssinian,14-17,"Active, Energetic, Independent, Intelligent, G...",Egypt,EG,EG,"Medium-sized, elegant cat with a distinctive t...",Short-haired,"{'imperial': '8-12', 'metric': '3.6-5.4'}","{'imperial': '10-12', 'metric': '25-30'}"
1,Aegean,9-12,"Affectionate, Social, Intelligent, Playful, Ac...",Greece,GR,GR,Medium-sized natural breed with a semi-longhai...,Semi-longhair,"{'imperial': '9-12', 'metric': '4-5.4'}","{'imperial': '9-11', 'metric': '23-28'}"
2,American Bobtail,11-15,"Intelligent, Interactive, Lively, Playful, Sen...",United States,US,US,Medium to large cat with a distinctive natural...,Short/Medium-hair,"{'imperial': '12-16', 'metric': '5.4-7.3'}","{'imperial': '11-14', 'metric': '28-36'}"
3,American Curl,12-16,"Affectionate, Curious, Intelligent, Interactiv...",United States,US,US,Medium-sized cat with distinctive backward-cur...,Short/Long-hair,"{'imperial': '7-11', 'metric': '3.2-5'}","{'imperial': '9-12', 'metric': '23-30'}"
4,American Ringtail,12-18,"Friendly, Affectionate, Intelligent, Active, P...",United States,US,US,Medium-sized cat with distinctive curled tail ...,Short/Long-hair,"{'imperial': '7-15', 'metric': '3.2-6.8'}","{'imperial': '9-12', 'metric': '23-30'}"


In [11]:
# Mostramos los diferentes valores que contiene la columna life_span
df_cat_analysis["life_span"].unique()

<StringArray>
['14-17',  '9-12', '11-15', '12-16', '12-18', '15-20', '14-18', '12-15',
 '12-14', '12-20', '14-20', '12-17', '15-18', '10-15', '10-14',     '1',
 '14-16',  '8-14',     '5', '13-16',  '9-15', '10-12',     '4', '11-16',
 '11-14', '14-15', '13-15', '11-18', '14-19']
Length: 29, dtype: str

In [12]:
# Buscamos registros de life_span que no contienen un rango
df_cat_analysis[
    ~df_cat_analysis["life_span"].str.contains("-")
][["name", "life_span"]]

,name,life_span
24,Caracat,1
40,Dwelf,5
42,Elf Cat,5
56,Kellas Cat,1
68,Napoleon Longhair,4
75,Pantherette,1
92,Singapura Tabby,1


In [13]:
# Creamos el filtro para los registros cuyo life_span no contiene un rango
life_span_single = ~df_cat_analysis["life_span"].str.contains("-")

# Mostramos información adicional de estos registros
df_cat_analysis.loc[
    life_span_single,
    [
        "name",
        "life_span",
        "origin",
        "description",

    ]
]

,name,life_span,origin,description
24,Caracat,1,NaN,Large hybrid between caracal and domestic cat....
40,Dwelf,5,NaN,"Small, hairless cat with curled ears and short..."
42,Elf Cat,5,NaN,Medium-sized hairless cat with distinctive cur...
56,Kellas Cat,1,NaN,"Large, melanistic wild-domestic hybrid found i..."
68,Napoleon Longhair,4,NaN,Small to medium cat with short legs and long p...
75,Pantherette,1,NaN,"Medium to large, muscular black cat bred to re..."
92,Singapura Tabby,1,NaN,"Small, muscular cat with tabby pattern variati..."


In [14]:
# Mostramos el contenido completo de las columnas de texto
pd.set_option("display.max_colwidth", None)

In [15]:
# Restauramos la visualización por defecto
pd.reset_option("display.max_colwidth")

In [16]:
# Creamos el filtro para los registros cuyo life_span contiene un rango
life_span_valid = df_cat_analysis["life_span"].str.contains("-")

# Mostramos información adicional de estos registros
df_cat_analysis.loc[
    life_span_valid,
    [
    "name",
    "life_span",
    "temperament",
    "origin",
    "country_codes",
    "country_code",
    "description",
    "breed_group",
    "weight",
    "height"
    ]
]

,name,life_span,temperament,origin,country_codes,country_code,description,breed_group,weight,height
0,Abyssinian,14-17,"Active, Energetic, Independent, Intelligent, G...",Egypt,EG,EG,"Medium-sized, elegant cat with a distinctive t...",Short-haired,"{'imperial': '8-12', 'metric': '3.6-5.4'}","{'imperial': '10-12', 'metric': '25-30'}"
1,Aegean,9-12,"Affectionate, Social, Intelligent, Playful, Ac...",Greece,GR,GR,Medium-sized natural breed with a semi-longhai...,Semi-longhair,"{'imperial': '9-12', 'metric': '4-5.4'}","{'imperial': '9-11', 'metric': '23-28'}"
2,American Bobtail,11-15,"Intelligent, Interactive, Lively, Playful, Sen...",United States,US,US,Medium to large cat with a distinctive natural...,Short/Medium-hair,"{'imperial': '12-16', 'metric': '5.4-7.3'}","{'imperial': '11-14', 'metric': '28-36'}"
3,American Curl,12-16,"Affectionate, Curious, Intelligent, Interactiv...",United States,US,US,Medium-sized cat with distinctive backward-cur...,Short/Long-hair,"{'imperial': '7-11', 'metric': '3.2-5'}","{'imperial': '9-12', 'metric': '23-30'}"
4,American Ringtail,12-18,"Friendly, Affectionate, Intelligent, Active, P...",United States,US,US,Medium-sized cat with distinctive curled tail ...,Short/Long-hair,"{'imperial': '7-15', 'metric': '3.2-6.8'}","{'imperial': '9-12', 'metric': '23-30'}"
...,...,...,...,...,...,...,...,...,...,...
102,Toyger,12-15,"Playful, Social, Intelligent, Friendly, Outgoi...",United States,US,US,"Medium to large, muscular cat with distinctive...",Short-haired,"{'imperial': '10-15', 'metric': '4.5-6.8'}","{'imperial': '11-14', 'metric': '28-36'}"
103,Turkish Angora,12-18,"Affectionate, Agile, Clever, Gentle, Intellige...",Turkey,TR,TR,"Medium-sized, elegant cat with long silky coat...",Long-haired,"{'imperial': '7-14', 'metric': '3.2-6.4'}","{'imperial': '10-14', 'metric': '25-36'}"
104,Turkish Van,12-17,"Agile, Intelligent, Loyal, Playful, Energetic,...",Turkey,TR,TR,"Large, muscular cat with distinctive Van patte...",Long-haired,"{'imperial': '10-20', 'metric': '4.5-9.1'}","{'imperial': '13-17', 'metric': '33-43'}"
105,Ukrainian Levkoy,12-15,"Affectionate, Relaxed, Playful, Curious, Intel...",Ukraine,UA,UA,"Medium-sized, hairless cat with distinctive fo...",Hairless,"{'imperial': '8-12', 'metric': '3.6-5.4'}","{'imperial': '10-12', 'metric': '25-30'}"


Se detectaron 7 registros con anomalías estructurales. Se compararon con los registros correctamente formateados para determinar si el problema afectaba únicamente a variables concretas o al registro completo.

In [17]:
#Vamos a crear life_span_min y life_span_max a partir de la columna life_span. Para ello, primero vamos a separar los valores de life_span en dos columnas nuevas.
# Separamos life_span en dos columnas usando el guion
life_span_split = (
    df_cat_analysis.loc[life_span_valid, "life_span"]
    .str.split("-", expand=True)
)

life_span_split.head()


,0,1
0,14,17
1,9,12
2,11,15
3,12,16
4,12,18


In [18]:
# Añadimos las columnas de esperanza de vida mínima y máxima
# y convertimos sus valores a tipo numérico
df_cat_analysis.loc[life_span_valid, "life_span_min"] = (
    pd.to_numeric(life_span_split[0])
)

df_cat_analysis.loc[life_span_valid, "life_span_max"] = (
    pd.to_numeric(life_span_split[1])
)

In [19]:
# Comprobamos las nuevas columnas
df_cat_analysis[
    ["name", "life_span", "life_span_min", "life_span_max"]
].head()

,name,life_span,life_span_min,life_span_max
0,Abyssinian,14-17,14.0,17.0
1,Aegean,9-12,9.0,12.0
2,American Bobtail,11-15,11.0,15.0
3,American Curl,12-16,12.0,16.0
4,American Ringtail,12-18,12.0,18.0


In [20]:
df_cat_analysis.head()

,name,life_span,temperament,origin,country_codes,country_code,description,breed_group,weight,height,life_span_min,life_span_max
0,Abyssinian,14-17,"Active, Energetic, Independent, Intelligent, G...",Egypt,EG,EG,"Medium-sized, elegant cat with a distinctive t...",Short-haired,"{'imperial': '8-12', 'metric': '3.6-5.4'}","{'imperial': '10-12', 'metric': '25-30'}",14.0,17.0
1,Aegean,9-12,"Affectionate, Social, Intelligent, Playful, Ac...",Greece,GR,GR,Medium-sized natural breed with a semi-longhai...,Semi-longhair,"{'imperial': '9-12', 'metric': '4-5.4'}","{'imperial': '9-11', 'metric': '23-28'}",9.0,12.0
2,American Bobtail,11-15,"Intelligent, Interactive, Lively, Playful, Sen...",United States,US,US,Medium to large cat with a distinctive natural...,Short/Medium-hair,"{'imperial': '12-16', 'metric': '5.4-7.3'}","{'imperial': '11-14', 'metric': '28-36'}",11.0,15.0
3,American Curl,12-16,"Affectionate, Curious, Intelligent, Interactiv...",United States,US,US,Medium-sized cat with distinctive backward-cur...,Short/Long-hair,"{'imperial': '7-11', 'metric': '3.2-5'}","{'imperial': '9-12', 'metric': '23-30'}",12.0,16.0
4,American Ringtail,12-18,"Friendly, Affectionate, Intelligent, Active, P...",United States,US,US,Medium-sized cat with distinctive curled tail ...,Short/Long-hair,"{'imperial': '7-15', 'metric': '3.2-6.8'}","{'imperial': '9-12', 'metric': '23-30'}",12.0,18.0


axis=0 → calcula por columnas.
axis=1 → calcula por filas. ← Esto queremos.

In [21]:
# Calculamos la esperanza de vida media de cada raza
df_cat_analysis["life_span_mean"] = df_cat_analysis[
    ["life_span_min", "life_span_max"]
].mean(axis=1)

In [22]:
# Comprobamos las nuevas columnas
df_cat_analysis[
    ["name", "life_span", "life_span_min", "life_span_max", "life_span_mean"]
].head()

,name,life_span,life_span_min,life_span_max,life_span_mean
0,Abyssinian,14-17,14.0,17.0,15.5
1,Aegean,9-12,9.0,12.0,10.5
2,American Bobtail,11-15,11.0,15.0,13.0
3,American Curl,12-16,12.0,16.0,14.0
4,American Ringtail,12-18,12.0,18.0,15.0


In [23]:
# Mostramos los diferentes valores que contiene la columna life_span
df_cat_analysis["life_span_mean"].unique()

array([15.5, 10.5, 13. , 14. , 15. , 17.5, 16. , 13.5, 17. , 14.5, 16.5,
       12.5, 12. ,  nan, 11. ])

In [24]:
#nan corresponde anomalias, eliminamos los registros del dataframe.
# Creamos un DataFrame sin los registros anómalos de life_span
df_cat_clean = df_cat_analysis.loc[life_span_valid].copy()

# Comprobamos las dimensiones
print(df_cat_clean.shape)

(100, 13)


In [25]:
# Separamos las características de temperament por comas
temperaments = (
    df_cat_clean["temperament"]
    .str.split(",")
    .explode()
    .str.strip()
)


# Mostramos los diferentes temperamentos
temperaments.unique()

<StringArray>
[                                       'Active',
                                     'Energetic',
                                   'Independent',
                                   'Intelligent',
                                        'Gentle',
                                       'Curious',
                                       'Playful',
                                  'Affectionate',
                                        'Social',
                                     'Adaptable',
                                   'Interactive',
                                        'Lively',
                                     'Sensitive',
                                       'Devoted',
                                      'Friendly',
                                    'Easy Going',
                                          'Calm',
                                         'Loyal',
                                      'Sensible',
                                    

In [26]:
# Contamos cuántas veces aparece cada temperamento
temperament_counts = temperaments.value_counts()

temperament_counts

temperament
Affectionate            74
Intelligent             69
Playful                 69
Social                  54
Active                  44
                        ..
Easygoing                1
Docile                   1
Silly                    1
Easily leash-trained     1
Tenacious                1
Name: count, Length: 63, dtype: int64

Esto ya nos da una pista bastante potente: en las 100 razas válidas, los temperamentos dominantes aparecen muchísimas veces

In [27]:
# Mostramos todos los temperamentos y su frecuencia
for temperament, count in temperament_counts.items():
    print(f"{temperament}: {count}")

Affectionate: 74
Intelligent: 69
Playful: 69
Social: 54
Active: 44
Gentle: 35
Loyal: 34
Curious: 27
Friendly: 21
Lively: 14
Calm: 14
Sweet: 14
Independent: 12
Quiet: 11
Easy Going: 10
Agile: 10
Interactive: 9
Energetic: 7
Adaptable: 7
Athletic: 7
Loving: 7
Devoted: 6
Dependent: 5
Demanding: 4
Patient: 4
Clever: 4
Vocal: 4
Outgoing: 4
Relaxed: 3
Mischievous: 3
Varies widely - typically Friendly: 3
Highly intelligent: 3
Sensitive: 2
Alert: 2
Peaceful: 2
Confident: 2
Trainable: 2
Inquisitive: 2
Sociable: 2
People-oriented: 2
Sweet-tempered: 2
Reserved: 2
Adventurous: 2
Sensible: 1
Fun-loving: 1
Tolerant: 1
Observant: 1
Highly interactive: 1
Talkative: 1
Warm: 1
Expressive: 1
Personable: 1
Cheerful: 1
Charming: 1
Extremely affectionate: 1
Shy: 1
Sedate: 1
Varies by individual - typically Affectionate: 1
Easygoing: 1
Docile: 1
Silly: 1
Easily leash-trained: 1
Tenacious: 1


In [28]:
# Unificamos variaciones claras de una misma característica
temperament_replace = {
    "Easygoing": "Easy Going",
    "Highly intelligent": "Intelligent",
    "Highly interactive": "Interactive",
    "Extremely affectionate": "Affectionate",
    "Sociable": "Social"
}

In [29]:
# Probamos la unificación sin modificar el DataFrame
temperaments_clean = temperaments.replace(temperament_replace)

print("Temperamentos antes:", temperaments.nunique())
print("Temperamentos después:", temperaments_clean.nunique())

Temperamentos antes: 63
Temperamentos después: 58


In [30]:
# Contamos de nuevo la frecuencia después de unificar los temperamentos
temperament_counts_clean = temperaments_clean.value_counts()

for temperament, count in temperament_counts_clean.items():
    print(f"{temperament}: {count}")

Affectionate: 75
Intelligent: 72
Playful: 69
Social: 56
Active: 44
Gentle: 35
Loyal: 34
Curious: 27
Friendly: 21
Lively: 14
Calm: 14
Sweet: 14
Independent: 12
Easy Going: 11
Quiet: 11
Interactive: 10
Agile: 10
Energetic: 7
Adaptable: 7
Athletic: 7
Loving: 7
Devoted: 6
Dependent: 5
Demanding: 4
Patient: 4
Clever: 4
Vocal: 4
Outgoing: 4
Relaxed: 3
Mischievous: 3
Varies widely - typically Friendly: 3
Sensitive: 2
Alert: 2
Peaceful: 2
Confident: 2
Trainable: 2
Inquisitive: 2
People-oriented: 2
Sweet-tempered: 2
Reserved: 2
Adventurous: 2
Sensible: 1
Fun-loving: 1
Tolerant: 1
Observant: 1
Talkative: 1
Warm: 1
Expressive: 1
Personable: 1
Cheerful: 1
Charming: 1
Shy: 1
Sedate: 1
Varies by individual - typically Affectionate: 1
Docile: 1
Silly: 1
Easily leash-trained: 1
Tenacious: 1


In [31]:
# Añadimos las etiquetas que indican el temperamento típico
temperament_replace.update({
    "Varies widely - typically Friendly": "Friendly",
    "Varies by individual - typically Affectionate": "Affectionate"
})

# Volvemos a aplicar la normalización
temperaments_clean = temperaments.replace(temperament_replace)

print("Temperamentos diferentes:", temperaments_clean.nunique())

Temperamentos diferentes: 56


In [32]:
# Mostramos todos los temperamentos distintos ya normalizados
for temperament in sorted(temperaments_clean.unique()):
    print(temperament)

Active
Adaptable
Adventurous
Affectionate
Agile
Alert
Athletic
Calm
Charming
Cheerful
Clever
Confident
Curious
Demanding
Dependent
Devoted
Docile
Easily leash-trained
Easy Going
Energetic
Expressive
Friendly
Fun-loving
Gentle
Independent
Inquisitive
Intelligent
Interactive
Lively
Loving
Loyal
Mischievous
Observant
Outgoing
Patient
Peaceful
People-oriented
Personable
Playful
Quiet
Relaxed
Reserved
Sedate
Sensible
Sensitive
Shy
Silly
Social
Sweet
Sweet-tempered
Talkative
Tenacious
Tolerant
Trainable
Vocal
Warm


In [33]:
temperament_categories = {
    "Social / Affectionate": [
        "Affectionate",
        "Charming",
        "Devoted",
        "Friendly",
        "Gentle",
        "Loving",
        "Loyal",
        "Outgoing",
        "People-oriented",
        "Personable",
        "Social",
        "Sweet",
        "Sweet-tempered",
        "Warm"
    ],

    "Active / Playful": [
        "Active",
        "Adventurous",
        "Agile",
        "Athletic",
        "Energetic",
        "Fun-loving",
        "Lively",
        "Mischievous",
        "Playful",
        "Silly"
    ]
}

In [34]:
temperament_categories.keys()

dict_keys(['Social / Affectionate', 'Active / Playful'])

In [35]:
temperament_categories.update({
    "Intelligent / Curious": [
        "Alert",
        "Clever",
        "Curious",
        "Inquisitive",
        "Intelligent",
        "Observant",
        "Sensible",
        "Tenacious",
        "Trainable"
    ],

    "Calm / Easygoing": [
        "Calm",
        "Docile",
        "Easy Going",
        "Gentle",
        "Patient",
        "Peaceful",
        "Quiet",
        "Relaxed",
        "Sedate",
        "Sweet-tempered",
        "Tolerant"
    ],

    "Independent / Reserved": [
        "Independent",
        "Reserved",
        "Shy"
    ],

    "Dependent / Interactive": [
        "Dependent",
        "Devoted",
        "Interactive",
        "People-oriented"
    ],

    "Expressive / Vocal": [
        "Expressive",
        "Talkative",
        "Vocal"
    ],

    "Confident / Strong personality": [
        "Confident",
        "Demanding",
        "Tenacious"
    ],

    "Adaptable": [
        "Adaptable",
        "Tolerant"
    ],

    "Trainable": [
        "Easily leash-trained",
        "Trainable"
    ]
})

In [36]:
temperament_categories.keys()

dict_keys(['Social / Affectionate', 'Active / Playful', 'Intelligent / Curious', 'Calm / Easygoing', 'Independent / Reserved', 'Dependent / Interactive', 'Expressive / Vocal', 'Confident / Strong personality', 'Adaptable', 'Trainable'])

In [37]:
categorized_temperaments = []

for category in temperament_categories.values():
    categorized_temperaments.extend(category)

len(set(categorized_temperaments))

54

In [38]:
missing_temperaments = set(temperaments_clean.unique()) - set(categorized_temperaments)

missing_temperaments

{'Cheerful', 'Sensitive'}

In [39]:
temperament_categories["Active / Playful"].append("Cheerful")
temperament_categories["Calm / Easygoing"].append("Sensitive")

In [40]:
categorized_temperaments = []

for category in temperament_categories.values():
    categorized_temperaments.extend(category)

len(set(categorized_temperaments))

56

In [41]:
df_cat_clean["breed_group"].unique()

<StringArray>
[         'Short-haired',         'Semi-longhair',     'Short/Medium-hair',
       'Short/Long-hair',       'Short/Semi-long',           'Long-haired',
              'Hairless',                'Hybrid',                   'Rex',
           'Medium-hair',    'Partially hairless', 'Hairless/Short-legged',
                'Varies',   'Short/Semi-longhair']
Length: 14, dtype: str

In [42]:
special_breed_groups = ["Rex", "Hybrid", "Varies"]

df_cat_clean[
    df_cat_clean["breed_group"].isin(special_breed_groups)
][["name", "breed_group", "description"]]

,name,breed_group,description
27,Chausie,Hybrid,"Large, athletic hybrid cat with a wild appeara..."
28,Chausie,Hybrid,"Large, athletic hybrid cat with wild appearanc..."
31,Cornish Rex,Rex,"Medium-sized, elegant cat with distinctive wav..."
34,Devon Rex,Rex,Small to medium cat with distinctive wavy coat...
49,German Rex,Rex,Medium-sized cat with wavy coat similar to oth...
60,LaPerm,Rex,Medium-sized cat with distinctive curly or wav...
79,Polydactyl,Varies,Domestic cat of any breed or type with extra t...
84,Safari,Hybrid,Large hybrid cat crossed from Geoffroy's cat a...
85,Savannah,Hybrid,"Large to very large tall spotted cat, a hybrid..."
87,Selkirk Rex,Rex,"Medium to large cat with plush, curly coat res..."


In [43]:
breed_group_replace = {
    "Short-haired": "Short",
    "Medium-hair": "Medium",
    "Long-haired": "Long",
    "Semi-longhair": "Long",
    "Short/Medium-hair": "Short / Medium",
    "Short/Long-hair": "Short / Long",
    "Short/Semi-long": "Short / Long",
    "Short/Semi-longhair": "Short / Long",
    "Hairless": "Hairless",
    "Partially hairless": "Hairless",
    "Hairless/Short-legged": "Hairless"
}
df_cat_clean["breed_group"].replace(breed_group_replace).value_counts(dropna=False)

breed_group
Short             44
Long              23
Short / Long      13
Hairless           7
Rex                6
Hybrid             4
Short / Medium     1
Medium             1
Varies             1
Name: count, dtype: int64

In [44]:
df_cat_clean["breed_group"] = (
    df_cat_clean["breed_group"]
    .replace(breed_group_replace)
)
df_cat_clean["breed_group"].unique()

<StringArray>
[         'Short',           'Long', 'Short / Medium',   'Short / Long',
       'Hairless',         'Hybrid',            'Rex',         'Medium',
         'Varies']
Length: 9, dtype: str

In [45]:
# Extraemos el peso en kg
df_cat_clean["weight_metric"] = (
    df_cat_clean["weight"].apply(lambda x: x["metric"])
)

In [46]:
df_cat_clean[
    ["name", "weight", "weight_metric"]
].head()

,name,weight,weight_metric
0,Abyssinian,"{'imperial': '8-12', 'metric': '3.6-5.4'}",3.6-5.4
1,Aegean,"{'imperial': '9-12', 'metric': '4-5.4'}",4-5.4
2,American Bobtail,"{'imperial': '12-16', 'metric': '5.4-7.3'}",5.4-7.3
3,American Curl,"{'imperial': '7-11', 'metric': '3.2-5'}",3.2-5
4,American Ringtail,"{'imperial': '7-15', 'metric': '3.2-6.8'}",3.2-6.8


In [47]:
# Comprobamos si hay pesos sin rango
df_cat_clean[
    ~df_cat_clean["weight_metric"].str.contains("-")
][["name", "weight_metric"]]

,name,weight_metric


In [48]:
# Separamos el peso mínimo y máximo
weight_split = (
    df_cat_clean["weight_metric"]
    .str.split("-", expand=True)
)

weight_split.head()

,0,1
0,3.6,5.4
1,4,5.4
2,5.4,7.3
3,3.2,5
4,3.2,6.8


In [49]:
# Creamos las columnas de peso mínimo y máximo en kg
df_cat_clean["weight_min"] = pd.to_numeric(weight_split[0])
df_cat_clean["weight_max"] = pd.to_numeric(weight_split[1])

In [50]:
df_cat_clean[
    ["name", "weight_metric", "weight_min", "weight_max"]
].head()

,name,weight_metric,weight_min,weight_max
0,Abyssinian,3.6-5.4,3.6,5.4
1,Aegean,4-5.4,4.0,5.4
2,American Bobtail,5.4-7.3,5.4,7.3
3,American Curl,3.2-5,3.2,5.0
4,American Ringtail,3.2-6.8,3.2,6.8


In [51]:
# Calculamos el peso medio en kg
df_cat_clean["weight_mean"] = df_cat_clean[
    ["weight_min", "weight_max"]
].mean(axis=1)

df_cat_clean[
    ["name", "weight_min", "weight_max", "weight_mean"]
].head()

,name,weight_min,weight_max,weight_mean
0,Abyssinian,3.6,5.4,4.50
1,Aegean,4.0,5.4,4.70
2,American Bobtail,5.4,7.3,6.35
3,American Curl,3.2,5.0,4.10
4,American Ringtail,3.2,6.8,5.00


In [52]:
animals = pd.read_csv("9t4d-g238_version_52594.csv")
animals.head()

,animal_id,name,outcome_type,outcome_subtype,animal_type,sex_upon_outcome,age_upon_outcome,breed,color,monthyear,datetime,date_of_birth
0,A705049,*Fancy,Transfer,Partner,Cat,Spayed Female,2 months,Domestic Shorthair Mix,Black,07-2015,2015-07-02T00:00:00-05:00,2015-04-05
1,A729515,Oreo,Transfer,Partner,Cat,Neutered Male,12 years,Domestic Shorthair Mix,Black/White,06-2016,2016-06-20T00:00:00-05:00,2004-06-19
2,A680862,NaN,Transfer,Partner,Bird,Unknown,2 weeks,Duck,Yellow/Black,06-2014,2014-06-10T00:00:00-05:00,2014-05-25
3,A714876,*Newton,Transfer,Partner,Cat,Neutered Male,3 years,Domestic Shorthair Mix,Orange Tabby/White,11-2015,2015-11-11T00:00:00-05:00,2012-10-29
4,A706507,NaN,Died,In Kennel,Cat,Unknown,5 days,Domestic Shorthair Mix,Black,06-2015,2015-06-30T00:00:00-05:00,2015-06-25


In [53]:
is_cat = animals["animal_type"] == "Cat"
cats = animals[is_cat]
cats["animal_type"].value_counts()

animal_type
Cat    69399
Name: count, dtype: int64

In [54]:
cats.info()

<class 'pandas.DataFrame'>
Index: 69399 entries, 0 to 173774
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   animal_id         69399 non-null  str  
 1   name              42473 non-null  str  
 2   outcome_type      69390 non-null  str  
 3   outcome_subtype   40606 non-null  str  
 4   animal_type       69399 non-null  str  
 5   sex_upon_outcome  69399 non-null  str  
 6   age_upon_outcome  69395 non-null  str  
 7   breed             69399 non-null  str  
 8   color             69399 non-null  str  
 9   monthyear         69399 non-null  str  
 10  datetime          69399 non-null  str  
 11  date_of_birth     69399 non-null  str  
dtypes: str(12)
memory usage: 6.9 MB


In [55]:
animals["outcome_type"].unique()

<StringArray>
[       'Transfer',            'Died',        'Adoption', 'Return to Owner',
         'Missing',        'Disposal',      'Euthanasia',               nan,
        'Relocate',       'Rto-Adopt',          'Stolen',            'Lost']
Length: 12, dtype: str

In [56]:
# Comprobamos cuántos gatos únicos hay
cats["animal_id"].nunique()

66286

In [57]:
# Vemos cuántos gatos hay en cada tipo de outcome
cats["outcome_type"].value_counts(dropna=False)

outcome_type
Adoption           35781
Transfer           26702
Return to Owner     2868
Euthanasia          2563
Died                1006
Rto-Adopt            249
Disposal             164
Missing               51
NaN                    9
Relocate               6
Name: count, dtype: int64

In [58]:
# Vemos los diferentes formatos de edad
cats["age_upon_outcome"].unique()

<StringArray>
[ '2 months',  '12 years',   '3 years',    '5 days',  '4 months',   '1 month',
   '2 years',   '5 years',  '10 years',   '2 weeks',  '8 months',    '1 year',
   '3 weeks',   '4 years',  '3 months',   '7 years', '10 months',    '3 days',
   '1 weeks',  '6 months',   '4 weeks', '11 months',  '9 months',  '7 months',
    '4 days',   '6 years',  '5 months',    '1 week',    '2 days',   '8 years',
   '5 weeks',   '9 years',  '17 years',  '13 years',  '15 years',   '0 years',
  '14 years',  '11 years',     '1 day',    '6 days',  '18 years',  '19 years',
  '16 years',  '20 years',  '22 years',  '-4 years',         nan]
Length: 47, dtype: str

In [59]:
# Creamos un DataFrame solo con los gatos adoptados
cats_adopted = cats[cats["outcome_type"] == "Adoption"].copy()

cats_adopted.shape

(35781, 12)

In [60]:
cats_adopted.head()

,animal_id,name,outcome_type,outcome_subtype,animal_type,sex_upon_outcome,age_upon_outcome,breed,color,monthyear,datetime,date_of_birth
15,A713929,Wingman,Adoption,NaN,Cat,Neutered Male,2 years,Siamese Mix,Seal Point,10-2015,2015-10-17T00:00:00-05:00,2013-05-15
79,A725759,NaN,Adoption,NaN,Cat,Neutered Male,2 months,Domestic Longhair Mix,Brown Tabby,05-2016,2016-05-06T00:00:00-05:00,2016-02-13
122,A721115,Cloud Burst,Adoption,NaN,Cat,Spayed Female,1 year,Domestic Shorthair Mix,Calico,06-2016,2016-06-26T00:00:00-05:00,2015-02-21
130,A713852,Peach,Adoption,NaN,Cat,Intact Female,2 months,Domestic Shorthair Mix,Orange Tabby/White,10-2015,2015-10-18T00:00:00-05:00,2015-08-14
141,A725756,Serenity,Adoption,NaN,Cat,Spayed Female,2 months,Domestic Longhair Mix,Black,05-2016,2016-05-06T00:00:00-05:00,2016-02-23


In [61]:
# Buscamos los IDs que aparecen más de una vez
duplicated_ids = cats_adopted[
    cats_adopted["animal_id"].duplicated(keep=False)
]

duplicated_ids["animal_id"].value_counts().head(10)

animal_id
A661049    5
A707669    4
A720883    4
A745494    4
A749086    4
A751091    4
A764518    4
A762275    4
A786435    4
A839910    4
Name: count, dtype: int64

In [62]:
# Vemos todas las adopciones del gato A839910
cats_adopted[
    cats_adopted["animal_id"] == "A839910"
].sort_values("datetime")

,animal_id,name,outcome_type,outcome_subtype,animal_type,sex_upon_outcome,age_upon_outcome,breed,color,monthyear,datetime,date_of_birth
131332,A839910,Melody,Adoption,NaN,Cat,Spayed Female,2 years,Siamese,Lynx Point,08-2021,2021-08-22T15:07:00,2019-07-29
134543,A839910,Melody,Adoption,NaN,Cat,Spayed Female,2 years,Siamese,Lynx Point,11-2021,2021-11-15T12:41:00,2019-07-29
135927,A839910,Melody,Adoption,NaN,Cat,Spayed Female,2 years,Siamese,Lynx Point,12-2021,2021-12-18T17:57:00,2019-07-29
100877,A839910,Melody,Adoption,NaN,Cat,Spayed Female,3 years,Siamese,Lynx Point,08-2022,2022-08-23T00:00:00-05:00,2019-07-29


In [63]:
# Un mismo animal_id puede tener varias adopciones.
# Cada fila representa un episodio de adopción, no un gato único.
# Por este motivo, no eliminamos los animal_id duplicados.

In [64]:
cats_adopted["sex_upon_outcome"].unique()

<StringArray>
['Neutered Male', 'Spayed Female', 'Intact Female', 'Intact Male', 'Unknown']
Length: 5, dtype: str

In [65]:
cats_adopted["sex_upon_outcome"].value_counts()

sex_upon_outcome
Spayed Female    17389
Neutered Male    16925
Intact Female      823
Intact Male        643
Unknown              1
Name: count, dtype: int64

In [66]:
# Mostramos la distribución del sexo en porcentaje
cats_adopted["sex_upon_outcome"].value_counts(normalize=True).map(
    lambda x: f"{x * 100:.2f}%"
)

sex_upon_outcome
Spayed Female    48.60%
Neutered Male    47.30%
Intact Female     2.30%
Intact Male       1.80%
Unknown           0.00%
Name: proportion, dtype: str

In [67]:
cats_adopted.columns

Index(['animal_id', 'name', 'outcome_type', 'outcome_subtype', 'animal_type',
       'sex_upon_outcome', 'age_upon_outcome', 'breed', 'color', 'monthyear',
       'datetime', 'date_of_birth'],
      dtype='str')

In [68]:
cats_adopted.head()

,animal_id,name,outcome_type,outcome_subtype,animal_type,sex_upon_outcome,age_upon_outcome,breed,color,monthyear,datetime,date_of_birth
15,A713929,Wingman,Adoption,NaN,Cat,Neutered Male,2 years,Siamese Mix,Seal Point,10-2015,2015-10-17T00:00:00-05:00,2013-05-15
79,A725759,NaN,Adoption,NaN,Cat,Neutered Male,2 months,Domestic Longhair Mix,Brown Tabby,05-2016,2016-05-06T00:00:00-05:00,2016-02-13
122,A721115,Cloud Burst,Adoption,NaN,Cat,Spayed Female,1 year,Domestic Shorthair Mix,Calico,06-2016,2016-06-26T00:00:00-05:00,2015-02-21
130,A713852,Peach,Adoption,NaN,Cat,Intact Female,2 months,Domestic Shorthair Mix,Orange Tabby/White,10-2015,2015-10-18T00:00:00-05:00,2015-08-14
141,A725756,Serenity,Adoption,NaN,Cat,Spayed Female,2 months,Domestic Longhair Mix,Black,05-2016,2016-05-06T00:00:00-05:00,2016-02-23


In [70]:
# Eliminamos los registros con sexo desconocido
cats_adopted = cats_adopted[
    cats_adopted["sex_upon_outcome"] != "Unknown"
].copy()
# Mostramos la distribución del sexo en porcentaje
cats_adopted["sex_upon_outcome"].value_counts(normalize=True).map(
    lambda x: f"{x * 100:.2f}%"
)

sex_upon_outcome
Spayed Female    48.60%
Neutered Male    47.30%
Intact Female     2.30%
Intact Male       1.80%
Name: proportion, dtype: str

El porcentaje de gatos castrados es tan bajo que no lo consideramos relevante a la hora de adoptarlos. Esto puede deberse a temas de salud de los animales o por temas económicos, ya que normalmente las protectoras de animales cuentan con un presupuesto bajo para este tipo de operaciones, que son bastante costosas (especialmente en hembras). Suelen requerir a los adoptantes que los castren como condición para la adopción. Además, es preferible castrar a los animales cuando son pequeños, ya que de mayores pueden sufrir algunas dolencias. Además, lo normal es que las protectoras tengan animales algo mayores, por lo que no los castran y dejan esa responsabilidad a las personas que los adopten.

In [71]:
# Eliminamos las columnas que no necesitamos para el análisis
cats_adopted = cats_adopted.drop(
    columns=["name", "outcome_type", "outcome_subtype", "animal_type"]
)

In [72]:
cats_adopted.head()

,animal_id,sex_upon_outcome,age_upon_outcome,breed,color,monthyear,datetime,date_of_birth
15,A713929,Neutered Male,2 years,Siamese Mix,Seal Point,10-2015,2015-10-17T00:00:00-05:00,2013-05-15
79,A725759,Neutered Male,2 months,Domestic Longhair Mix,Brown Tabby,05-2016,2016-05-06T00:00:00-05:00,2016-02-13
122,A721115,Spayed Female,1 year,Domestic Shorthair Mix,Calico,06-2016,2016-06-26T00:00:00-05:00,2015-02-21
130,A713852,Intact Female,2 months,Domestic Shorthair Mix,Orange Tabby/White,10-2015,2015-10-18T00:00:00-05:00,2015-08-14
141,A725756,Spayed Female,2 months,Domestic Longhair Mix,Black,05-2016,2016-05-06T00:00:00-05:00,2016-02-23


In [78]:
# Número de razas diferentes
cats_adopted["breed"].nunique()
# Vemos las razas más frecuentes
cats_adopted["breed"].value_counts()

breed
Domestic Shorthair Mix            15762
Domestic Shorthair                13760
Domestic Medium Hair Mix           1682
Domestic Medium Hair               1308
Domestic Longhair Mix               855
                                  ...  
Pixiebob Shorthair                    1
Siamese/Munchkin Shorthair            1
Domestic Shorthair/Oriental Sh        1
Oriental Sh                           1
Rex                                   1
Name: count, Length: 90, dtype: int64

In [74]:
# Comprobamos cuántos registros contienen "Mix" en la raza
cats_adopted["breed"].str.contains("Mix", case=False, na=False).value_counts()

breed
True     19654
False    16126
Name: count, dtype: int64

In [75]:
cats_adopted["breed"].str.contains(
    "Mix", case=False, na=False
).value_counts(normalize=True).map(
    lambda x: f"{x * 100:.2f}%"
)

breed
True     54.93%
False    45.07%
Name: proportion, dtype: str

In [79]:
# Vemos las razas diferentes que contienen "Mix"
cats_adopted.loc[
    cats_adopted["breed"].str.contains("Mix", case=False, na=False),
    "breed"
].unique()

<StringArray>
[                'Siamese Mix',       'Domestic Longhair Mix',
      'Domestic Shorthair Mix',    'Domestic Medium Hair Mix',
      'American Shorthair Mix',                'Snowshoe Mix',
                    'Manx Mix',              'Maine Coon Mix',
              'Abyssinian Mix',                 'Persian Mix',
            'Russian Blue Mix',                'Balinese Mix',
                 'Ragdoll Mix',                  'Angora Mix',
                  'Bengal Mix',               'Himalayan Mix',
        'Japanese Bobtail Mix',               'Tonkinese Mix',
       'British Shorthair Mix',    'Norwegian Forest Cat Mix',
      'Munchkin Shorthair Mix',               'Devon Rex Mix',
                  'Cymric Mix',             'Turkish Van Mix',
       'American Wirehair Mix',      'Pixiebob Shorthair Mix',
                  'Bombay Mix',                'Javanese Mix',
           'Scottish Fold Mix',               'Chartreux Mix',
            'Havana Brown Mix',          

In [80]:
# Buscamos razas que contienen "/"
cats_adopted[
    cats_adopted["breed"].str.contains("/", na=False)
]["breed"].value_counts()

breed
Siamese/Domestic Shorthair                  12
Domestic Longhair/Persian                    6
Siamese/Japanese Bobtail                     3
Manx/Domestic Shorthair                      2
Domestic Medium Hair/Maine Coon              2
Domestic Shorthair/Abyssinian                2
Bengal/Domestic Shorthair                    2
Domestic Shorthair/Domestic Shorthair        2
Domestic Medium Hair/Manx                    2
Domestic Longhair/Domestic Longhair          2
Domestic Shorthair/Manx                      2
Domestic Shorthair/Domestic Medium Hair      1
Manx/Siamese                                 1
Manx/Domestic Longhair                       1
Domestic Medium Hair/Siamese                 1
Domestic Shorthair/Maine Coon                1
Scottish Fold/Domestic Shorthair             1
American Curl Shorthair/Siamese              1
Japanese Bobtail/American Curl Shorthair     1
Domestic Shorthair/Siamese                   1
Snowshoe/Domestic Shorthair                  1
Ragdoll

In [81]:
# Creamos una columna auxiliar con la raza base
cats_adopted["breed_base"] = (
    cats_adopted["breed"]
    .str.replace(" Mix", "", regex=False)
)

In [82]:
# Comprobamos la nueva columna
cats_adopted[
    ["breed", "breed_base"]
].drop_duplicates().head(20)

,breed,breed_base
15,Siamese Mix,Siamese
79,Domestic Longhair Mix,Domestic Longhair
122,Domestic Shorthair Mix,Domestic Shorthair
553,Domestic Medium Hair Mix,Domestic Medium Hair
1211,American Shorthair Mix,American Shorthair
1265,Domestic Shorthair,Domestic Shorthair
2044,Domestic Shorthair/Domestic Medium Hair,Domestic Shorthair/Domestic Medium Hair
2184,Snowshoe Mix,Snowshoe
2697,Bengal,Bengal
3021,Manx Mix,Manx


In [83]:
# Quitamos " Mix" y separamos las razas por "/"
breed_split = (
    cats_adopted["breed"]
    .str.replace(" Mix", "", regex=False)
    .str.split("/", n=1, expand=True)
)

# Creamos las dos columnas
cats_adopted["breed_base_1"] = breed_split[0]
cats_adopted["breed_base_2"] = breed_split[1]

In [86]:
cats_adopted[
    ["breed", "breed_base_1", "breed_base_2"]
].drop_duplicates().head(10)

,breed,breed_base_1,breed_base_2
15,Siamese Mix,Siamese,NaN
79,Domestic Longhair Mix,Domestic Longhair,NaN
122,Domestic Shorthair Mix,Domestic Shorthair,NaN
553,Domestic Medium Hair Mix,Domestic Medium Hair,NaN
1211,American Shorthair Mix,American Shorthair,NaN
1265,Domestic Shorthair,Domestic Shorthair,NaN
2044,Domestic Shorthair/Domestic Medium Hair,Domestic Shorthair,Domestic Medium Hair
2184,Snowshoe Mix,Snowshoe,NaN
2697,Bengal,Bengal,NaN
3021,Manx Mix,Manx,NaN


In [ ]:
cats_adopted

,animal_id,sex_upon_outcome,age_upon_outcome,breed,color,monthyear,datetime,date_of_birth,breed_base,breed_base_1,breed_base_2
15,A713929,Neutered Male,2 years,Siamese Mix,Seal Point,10-2015,2015-10-17T00:00:00-05:00,2013-05-15,Siamese,Siamese,NaN
79,A725759,Neutered Male,2 months,Domestic Longhair Mix,Brown Tabby,05-2016,2016-05-06T00:00:00-05:00,2016-02-13,Domestic Longhair,Domestic Longhair,NaN
122,A721115,Spayed Female,1 year,Domestic Shorthair Mix,Calico,06-2016,2016-06-26T00:00:00-05:00,2015-02-21,Domestic Shorthair,Domestic Shorthair,NaN
130,A713852,Intact Female,2 months,Domestic Shorthair Mix,Orange Tabby/White,10-2015,2015-10-18T00:00:00-05:00,2015-08-14,Domestic Shorthair,Domestic Shorthair,NaN
141,A725756,Spayed Female,2 months,Domestic Longhair Mix,Black,05-2016,2016-05-06T00:00:00-05:00,2016-02-23,Domestic Longhair,Domestic Longhair,NaN
...,...,...,...,...,...,...,...,...,...,...,...
173761,A927303,Spayed Female,1 year,Domestic Shorthair,Tortie,05-2025,2025-05-04T17:13:00,2024-03-25,Domestic Shorthair,Domestic Shorthair,NaN
173765,A927840,Neutered Male,2 months,Domestic Longhair,Orange,05-2025,2025-05-03T15:44:00,2025-03-02,Domestic Longhair,Domestic Longhair,NaN
173766,A929063,Neutered Male,2 years,Domestic Shorthair,White/Orange Tabby,05-2025,2025-05-02T12:08:00,2023-04-24,Domestic Shorthair,Domestic Shorthair,NaN
173768,A929103,Spayed Female,1 year,Domestic Shorthair,Blue Tabby,05-2025,2025-05-03T12:20:00,2023-10-25,Domestic Shorthair,Domestic Shorthair,NaN


In [91]:
# Vemos cuántos colores diferentes tenemos
cats_adopted["color"].nunique()

220

In [92]:
# Vemos los colores más frecuentes
cats_adopted["color"].value_counts().head(20)

color
Brown Tabby           5490
Black                 4412
Black/White           3217
Brown Tabby/White     2951
Orange Tabby          2596
Tortie                1637
Calico                1609
Blue Tabby            1511
Orange Tabby/White    1458
Blue                  1307
Torbie                1242
Blue/White             959
Blue Tabby/White       800
Cream Tabby            724
White/Black            610
Lynx Point             522
White/Brown Tabby      403
Cream Tabby/White      347
Seal Point             302
Flame Point            262
Name: count, dtype: int64

In [93]:
# Comprobamos cuántos colores son combinaciones
cats_adopted["color"].str.contains("/", na=False).value_counts()

color
False    22910
True     12870
Name: count, dtype: int64

In [94]:
# Separamos las combinaciones de colores por "/"
color_split = (
    cats_adopted["color"]
    .str.split("/", n=1, expand=True)
)

# Creamos las dos columnas
cats_adopted["color_1"] = color_split[0]
cats_adopted["color_2"] = color_split[1]

In [95]:
cats_adopted[
    ["color", "color_1", "color_2"]
].drop_duplicates().head(20)

,color,color_1,color_2
15,Seal Point,Seal Point,NaN
79,Brown Tabby,Brown Tabby,NaN
122,Calico,Calico,NaN
130,Orange Tabby/White,Orange Tabby,White
141,Black,Black,NaN
142,White/Black,White,Black
181,Cream Tabby,Cream Tabby,NaN
197,Tortie,Tortie,NaN
411,Brown Tabby/White,Brown Tabby,White
497,Black/White,Black,White


In [96]:
# Comprobamos cuántos valores diferentes tenemos después de separar los colores
print("Color 1:", cats_adopted["color_1"].nunique())
print("Color 2:", cats_adopted["color_2"].nunique())
# Vemos los valores únicos de color_1
cats_adopted["color_1"].value_counts()

Color 1: 46
Color 2: 36


color_1
Brown Tabby          8512
Black                7690
Orange Tabby         4066
Blue Tabby           2326
Blue                 2277
White                2006
Calico               1779
Tortie               1774
Torbie               1519
Cream Tabby          1073
Lynx Point            596
Seal Point            362
Flame Point           267
Gray Tabby            237
Black Smoke           181
Black Tabby           144
Tortie Point          127
Orange                119
Silver Tabby           98
Lilac Point            98
Blue Point             78
Gray                   73
Chocolate Point        62
Cream                  58
Blue Cream             52
Calico Point           42
Brown                  36
Chocolate              23
Agouti                 19
Silver Lynx Point      17
Blue Smoke             15
Brown Tiger            12
Silver                 10
Buff                    9
Orange Tiger            4
Yellow                  3
Tricolor                3
Tan                     2
Apri

In [97]:
# Separamos la edad en valor y unidad
age_split = cats_adopted["age_upon_outcome"].str.split(" ", expand=True)

age_split.head()

,0,1
15,2,years
79,2,months
122,1,year
130,2,months
141,2,months


In [98]:
# Guardamos el valor numérico de la edad
cats_adopted["age_value"] = pd.to_numeric(age_split[0])

# Guardamos la unidad de la edad
cats_adopted["age_unit"] = age_split[1]

In [99]:
cats_adopted[
    ["age_upon_outcome", "age_value", "age_unit"]
].head(10)

,age_upon_outcome,age_value,age_unit
15,2 years,2,years
79,2 months,2,months
122,1 year,1,year
130,2 months,2,months
141,2 months,2,months
142,2 months,2,months
162,2 months,2,months
181,2 months,2,months
197,2 years,2,years
251,2 years,2,years


In [100]:
# Comprobamos las unidades de edad
cats_adopted["age_unit"].value_counts(dropna=False)

age_unit
months    24630
years      6589
year       2991
month      1567
weeks         3
Name: count, dtype: int64

In [101]:
# Convertimos todas las edades a meses
def age_to_months(row):
    if row["age_unit"] in ["year", "years"]:
        return row["age_value"] * 12
    elif row["age_unit"] in ["month", "months"]:
        return row["age_value"]
    elif row["age_unit"] == "weeks":
        return row["age_value"] / 4

cats_adopted["age_months"] = cats_adopted.apply(age_to_months, axis=1)

In [102]:
cats_adopted[
    ["age_upon_outcome", "age_value", "age_unit", "age_months"]
].head(10)

,age_upon_outcome,age_value,age_unit,age_months
15,2 years,2,years,24.0
79,2 months,2,months,2.0
122,1 year,1,year,12.0
130,2 months,2,months,2.0
141,2 months,2,months,2.0
142,2 months,2,months,2.0
162,2 months,2,months,2.0
181,2 months,2,months,2.0
197,2 years,2,years,24.0
251,2 years,2,years,24.0


In [103]:
# Vemos las estadísticas de la edad en meses
cats_adopted["age_months"].describe()

count    35780.000000
mean        13.562598
std         26.677873
min        -48.000000
25%          2.000000
50%          3.000000
75%         12.000000
max        228.000000
Name: age_months, dtype: float64

In [104]:
# Revisamos los registros con una edad imposible
cats_adopted[cats_adopted["age_months"] < 0]

,animal_id,sex_upon_outcome,age_upon_outcome,breed,color,monthyear,datetime,date_of_birth,breed_base,breed_base_1,breed_base_2,color_1,color_2,age_value,age_unit,age_months
91562,A783723,Intact Female,-4 years,Domestic Shorthair,White/Orange,11-2018,2018-11-12T11:53:00,2023-02-07,Domestic Shorthair,Domestic Shorthair,NaN,White,Orange,-4,years,-48.0


In [105]:
# Corregimos el registro con el error de escritura en la edad
mask_age_error = cats_adopted["animal_id"] == "A783723"

cats_adopted.loc[mask_age_error, "age_upon_outcome"] = "4 years"
cats_adopted.loc[mask_age_error, "age_value"] = 4
cats_adopted.loc[mask_age_error, "age_unit"] = "years"
cats_adopted.loc[mask_age_error, "age_months"] = 48

In [108]:
cats_adopted.loc[
    mask_age_error,
    ["animal_id", "age_upon_outcome", "age_value", "age_unit", "age_months"]
]

,animal_id,age_upon_outcome,age_value,age_unit,age_months
91562,A783723,4 years,4,years,48.0


In [ ]:
cats_adopted["age_months"].describe()

count    35780.000000
mean        13.565281
std         26.676509
min          0.000000
25%          2.000000
50%          3.000000
75%         12.000000
max        228.000000
Name: age_months, dtype: float64

Hay un total de 35 780 adopciones en este data frame. El gato con menos edad es de 0 meses, es decir, muy chico. El 25 % de los gatos adoptados tiene unos 2 meses nada más. La mediana es de 3 meses. Un 75 % de los gatos adoptados tiene 12 meses (un año) o menos, y el máximo de edad es de 19 años. La media es de algo más de 1 año, aunque este dato está influenciado por los gatos más sénior.

Lo verdaderamente llamativo es que el 75 % de los animales son adoptados a los 12 meses o antes, lo que nos hace concluir que la gente prefiere animales jóvenes.

In [110]:
# Vemos las edades más frecuentes en meses
cats_adopted["age_months"].value_counts().sort_index()

age_months
0.00          3
0.25          1
0.50          1
1.00       1568
2.00      12629
3.00       5458
4.00       2600
5.00       1320
6.00        987
7.00        491
8.00        433
9.00        250
10.00       302
11.00       160
12.00      2991
24.00      2576
36.00       954
48.00       587
60.00       483
72.00       338
84.00       312
96.00       374
108.00      207
120.00      275
132.00      117
144.00      113
156.00      100
168.00       66
180.00       46
192.00       21
204.00        9
216.00        7
228.00        1
Name: count, dtype: int64

In [111]:
# Vemos las edades más frecuentes en meses
cats_adopted["age_months"].value_counts().head(20)

age_months
2.0      12629
3.0       5458
12.0      2991
4.0       2600
24.0      2576
1.0       1568
5.0       1320
6.0        987
36.0       954
48.0       587
7.0        491
60.0       483
8.0        433
96.0       374
72.0       338
84.0       312
10.0       302
120.0      275
9.0        250
108.0      207
Name: count, dtype: int64

In [114]:
# Vemos las edades más frecuentes en porcentaje
cats_adopted["age_months"].value_counts(normalize=True).head(20).map(
    lambda x: f"{x * 100:.2f} %"
)

age_months
2.0      35.30 %
3.0      15.25 %
12.0      8.36 %
4.0       7.27 %
24.0      7.20 %
1.0       4.38 %
5.0       3.69 %
6.0       2.76 %
36.0      2.67 %
48.0      1.64 %
7.0       1.37 %
60.0      1.35 %
8.0       1.21 %
96.0      1.05 %
72.0      0.94 %
84.0      0.87 %
10.0      0.84 %
120.0     0.77 %
9.0       0.70 %
108.0     0.58 %
Name: proportion, dtype: str